# Step 5 (re-run) — LDA: multi-seed, rentang K diperlebar, dan preprocessing alternatif

Tujuan: (1) membuktikan apakah K=20 stabil antar-seed, (2) menguji apakah K optimal ada di ujung rentang, (3) menguji dampak kata boilerplate ("menggunakan", "metode", dst.) sebagai robustness check (Tahap 19).

Aturan: baseline dilaporkan apa adanya; konfigurasi alternatif hanya pembanding. Semua angka berasal dari run ini. Output memakai akhiran `_v2` agar file lama tidak tertimpa.

**Estimasi waktu (Colab CPU):** ~30-90 menit untuk grid penuh (8 K x 3 seed x 2 konfigurasi). Set `QUICK = True` untuk uji cepat.

In [ ]:
!pip install gensim -q

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from gensim import corpora
from gensim.models.coherencemodel import CoherenceModel
warnings.filterwarnings('ignore')

# ============ KONFIGURASI ============
QUICK = False                      # True = uji cepat (subset kecil)
SEEDS = [42, 123, 2024]
K_LIST = [5, 7, 10, 12, 15, 20, 25, 30]
TOP_N = 10
VECT_PARAMS = dict(ngram_range=(1, 3), max_df=0.85, min_df=5, max_features=5000)   # sama dengan REVISED
LDA_PARAMS  = dict(learning_method='batch', max_iter=20, n_jobs=1)                  # sama dengan REVISED
INPUT_CANDIDATES = ['df_model_final.csv', 'df_lda_topics.csv']
EXCLUDE_JOURNAL_PATTERNS = []      # ISI SETELAH KEPUTUSAN SCOPE, mis. ['IJASEIT']; kosong = tidak ada yang dibuang
DOMAIN_STOPWORDS = ['menggunakan','metode','berbasis','penerapan','implementasi','perancangan',
                    'rancang','bangun','pendekatan','perbandingan','berdasarkan','pengembangan',
                    'penentuan','studi','kasus','pengaruh','penggunaan']
# 'analisis' sengaja TIDAK dimasukkan karena bagian dari istilah 'analisis sentimen'.
# ======================================

if QUICK:
    SEEDS, K_LIST = [42, 123], [5, 10]

path = next(p for p in INPUT_CANDIDATES if os.path.exists(p))
df = pd.read_csv(path).dropna(subset=['tokens_joined']).reset_index(drop=True)
n_before = len(df)
for pat in EXCLUDE_JOURNAL_PATTERNS:
    df = df[~df['jurnal'].str.contains(pat, case=False, na=False, regex=False)]
df = df.reset_index(drop=True)
if QUICK:
    df = df.sample(4000, random_state=42).reset_index(drop=True)
print(f"Input: {path} | dokumen awal={n_before} | dipakai={len(df)} | dibuang oleh filter journal={n_before-len(df) if not QUICK else 'n/a (QUICK)'}")
print("Rentang tahun:", df['tahun'].min(), "-", df['tahun'].max())
docs = df['tokens_joined'].astype(str).tolist()

with open('lda_rerun_config_v2.json', 'w') as f:
    json.dump(dict(seeds=SEEDS, k_list=K_LIST, top_n=TOP_N, vect_params=VECT_PARAMS, lda_params=LDA_PARAMS,
                   exclude_journal_patterns=EXCLUDE_JOURNAL_PATTERNS, domain_stopwords=DOMAIN_STOPWORDS,
                   n_docs=len(df), input_file=path), f, indent=2)

## A. Diagnostik: komposisi jurnal dan kata generik
Periksa tabel jurnal untuk menentukan apakah ada sumber yang di luar cakupan ilmu komputer. Keputusan scope harus dibuat SEBELUM grid dijalankan (isi `EXCLUDE_JOURNAL_PATTERNS`, lalu jalankan ulang dari atas).

In [ ]:
print(df['jurnal'].value_counts().head(15).to_string())
print("\nJumlah jurnal:", df['jurnal'].nunique())

# document frequency unigram teratas -> kandidat stopword domain
cv = CountVectorizer(binary=True, min_df=5)
Xb = cv.fit_transform(docs)
dfreq = pd.Series(np.asarray(Xb.sum(axis=0)).ravel(), index=cv.get_feature_names_out()) / len(docs)
print("\n40 unigram dengan document frequency tertinggi (proporsi dokumen):")
print(dfreq.sort_values(ascending=False).head(40).round(3).to_string())
print("\nDF kata pada DOMAIN_STOPWORDS:")
print(dfreq.reindex(DOMAIN_STOPWORDS).round(3).to_string())

## B. Fungsi eksperimen (identik dengan REVISED, ditambah multi-seed)

In [ ]:
def build_representation(docs, stop_words):
    vec = CountVectorizer(stop_words=stop_words, **VECT_PARAMS)
    X = vec.fit_transform(docs)
    fn = np.array(vec.get_feature_names_out())
    analyzer = vec.build_analyzer()
    texts = [analyzer(d) for d in docs]
    dictionary = corpora.Dictionary(texts)
    return vec, X, fn, texts, dictionary

def run_grid(name, stop_words):
    vec, X, fn, texts, dictionary = build_representation(docs, stop_words)
    print(f"[{name}] vocab={len(fn)} | DTM={X.shape}")
    rows, store = [], {}
    for K in K_LIST:
        for seed in SEEDS:
            t0 = time.time()
            lda = LatentDirichletAllocation(n_components=K, random_state=seed, **LDA_PARAMS).fit(X)
            topics = [[fn[i] for i in comp.argsort()[::-1][:TOP_N]] for comp in lda.components_]
            flat = [w for t in topics for w in t]
            div = len(set(flat)) / len(flat)
            coh = CoherenceModel(topics=topics, texts=texts, dictionary=dictionary,
                                 coherence='c_v', processes=1).get_coherence()
            rows.append(dict(config=name, K=K, seed=seed, coherence_cv=coh, diversity=div,
                             perplexity=lda.perplexity(X)))
            if seed == SEEDS[0]:
                store[K] = dict(lda=lda, topics=topics)
            print(f"[{name}] K={K:>2} seed={seed:<5} coh={coh:.4f} div={div:.3f} ({time.time()-t0:.0f}s)")
            pd.DataFrame(rows).to_csv(f'lda_grid_{name}_v2.csv', index=False)   # simpan bertahap
    return pd.DataFrame(rows), store, dict(vec=vec, X=X, fn=fn)

def summarize(res):
    s = res.groupby('K').agg(coh_mean=('coherence_cv','mean'), coh_sd=('coherence_cv','std'),
                             div_mean=('diversity','mean'), div_sd=('diversity','std'),
                             ppl_mean=('perplexity','mean')).reset_index()
    z = lambda x: (x - x.mean()) / x.std(ddof=1) if len(x) > 1 else x*0
    s['combined_z'] = z(s['coh_mean']) + z(s['div_mean'])
    return s

## C. Baseline (tanpa stopword domain)

In [ ]:
res_base, store_base, rep_base = run_grid('baseline', None)
sum_base = summarize(res_base)
print(sum_base.round(4).to_string(index=False))
K_base = int(sum_base.loc[sum_base['combined_z'].idxmax(), 'K'])
print(f"\nK terpilih baseline (combined z-score tertinggi) = {K_base}")
if K_base == max(K_LIST): print("PERINGATAN: K terpilih berada di batas atas rentang; perlu K lebih besar.")

## D. Alternatif (stopword domain) — robustness check

In [ ]:
res_alt, store_alt, rep_alt = run_grid('domain_stopwords', DOMAIN_STOPWORDS)
sum_alt = summarize(res_alt)
print(sum_alt.round(4).to_string(index=False))
K_alt = int(sum_alt.loc[sum_alt['combined_z'].idxmax(), 'K'])
print(f"\nK terpilih alternatif = {K_alt}")

## E. Perbandingan baseline vs alternatif
**Catatan metodologis:** C_v dihitung pada korpus masing-masing konfigurasi (kata generik dibuang dari korpus alternatif), sehingga nilai antar-konfigurasi tidak sepenuhnya sebanding. Baca perbandingan ini bersama keterbacaan topik, bukan angka saja.

In [ ]:
cmp = sum_base.merge(sum_alt, on='K', suffixes=('_base','_alt'))
cmp['delta_coh'] = cmp['coh_mean_alt'] - cmp['coh_mean_base']
cmp['delta_div'] = cmp['div_mean_alt'] - cmp['div_mean_base']
print(cmp[['K','coh_mean_base','coh_mean_alt','delta_coh','div_mean_base','div_mean_alt','delta_div']].round(4).to_string(index=False))
cmp.to_csv('lda_baseline_vs_alt_v2.csv', index=False)
pd.concat([res_base, res_alt]).to_csv('lda_grid_all_v2.csv', index=False)

## F. Figure 3 (revisi): dua panel, error bar = SD antar-seed

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)
for s, lab, c in [(sum_base, 'Baseline', '#4C72B0'), (sum_alt, 'Domain stopwords removed', '#DD8452')]:
    axes[0].errorbar(s['K'], s['coh_mean'], yerr=s['coh_sd'], marker='o', capsize=3, label=lab, color=c, lw=1.2)
    axes[1].errorbar(s['K'], s['div_mean'], yerr=s['div_sd'], marker='s', capsize=3, label=lab, color=c, lw=1.2)
axes[0].set_ylabel('Coherence (C_v)'); axes[1].set_ylabel('Topic diversity')
for ax in axes:
    ax.set_xlabel('Number of topics (K)'); ax.grid(alpha=.3); ax.set_xticks(K_LIST)
axes[0].legend(frameon=False)
fig.suptitle(f'Figure 3. LDA coherence and topic diversity by number of topics (mean ± SD, {len(SEEDS)} seeds)', fontsize=11)
fig.tight_layout()
plt.savefig('figure3_lda_coherence_diversity_v2.png', dpi=300, bbox_inches='tight')
plt.savefig('figure3_lda_coherence_diversity_v2.pdf', bbox_inches='tight')
plt.show()

## G. Pilih konfigurasi final (keputusan manual, dicatat)
Kriteria yang ditetapkan sebelum melihat hasil: combined z-score dari rata-rata coherence dan diversity. Isi `FINAL_CONFIG` setelah membaca bagian E dan Figure 3, lalu tulis alasannya di Methods.

In [ ]:
FINAL_CONFIG = 'baseline'   # 'baseline' atau 'domain_stopwords'  <-- putuskan setelah melihat E
K_FINAL = None              # None = pakai K dengan combined z-score tertinggi pada konfigurasi tsb

store, rep, K_sel = (store_base, rep_base, K_base) if FINAL_CONFIG == 'baseline' else (store_alt, rep_alt, K_alt)
K_FINAL = K_FINAL or K_sel
lda_final, topics_final = store[K_FINAL]['lda'], store[K_FINAL]['topics']
X_final = rep['X']
print(f"Model final: config={FINAL_CONFIG}, K={K_FINAL}, seed={SEEDS[0]}")

dist = lda_final.transform(X_final)
df['dominant_topic'] = dist.argmax(axis=1)
df['topic_prob'] = dist.max(axis=1)

json.dump({str(i): t for i, t in enumerate(topics_final)}, open('lda_topics_final_v2.json', 'w'), indent=2, ensure_ascii=False)

size = df['dominant_topic'].value_counts().sort_index()
reps = (df.sort_values('topic_prob', ascending=False).groupby('dominant_topic').head(10)
          [['dominant_topic','topic_prob','tahun','jurnal','title_raw']].sort_values(['dominant_topic','topic_prob'], ascending=[True, False]))
reps.to_csv('lda_representative_docs_v2.csv', index=False)

for i, words in enumerate(topics_final):
    print(f"\n--- Topic {i} (n={size.get(i,0)}) ---\nTop words: {', '.join(words)}")
    for t in reps[reps.dominant_topic == i]['title_raw'].head(5): print('  -', t)

## H. Penamaan topik (MANUAL)
Baca top words dan dokumen representatif di atas, lalu isi nama. Jangan menyalin nama dari run sebelumnya tanpa memeriksa ulang.

In [ ]:
topic_names = {
    # 0: "....",
    # 1: "....",
}
assert len(topic_names) == K_FINAL, f"Lengkapi nama untuk {K_FINAL} topik (baru terisi {len(topic_names)})."
df['lda_topic_name'] = df['dominant_topic'].map(topic_names)

counts = pd.crosstab(df['tahun'], df['lda_topic_name'])
props = counts.div(counts.sum(axis=1), axis=0)
counts.to_csv('lda_topic_by_year_counts_v2.csv'); props.to_csv('lda_topic_by_year_proportions_v2.csv')
df.to_csv('df_lda_topics_v2.csv', index=False)
print(counts.sum(axis=1))   # cek: total per tahun (2025 = tahun tidak penuh)